In [1]:
from typing import Literal, TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from param import output
from pydantic import BaseModel, Field

llm = init_chat_model(
    model= "deepseek-flash",
    extra_body = {'thinking': {'type': 'disabled'}},
)

#定义State
class MyState(TypedDict):
    input:str
    output:str


#定义node
#llm/chat/weather/translation
def llm_node(state:MyState):
    prompt = f"""
    根据用户问题:{ state["input"] },进行意图识别,可选意图只有:[chat, weather, translate, unknown]
    聊天相关,返回 chat
    天气情况,返回 weather
    翻译相关,返回 translate
    未命中上述意图,返回 unknown
    只输出上面四个单词中的一个,不要输出任何其它内容
    """
    ai_message = llm.invoke(prompt)
    state["output"] = ai_message.content
    return state

def chat_node(state:MyState):
    state["output"] = "聊天..."
    return state


def weather_node(state:MyState):
    state["output"] = "天气..."
    return state

def translation_node(state:MyState):
    state["output"] = "翻译..."
    return state

def unknown_node(state:MyState):
    state["output"] = "不清楚..."
    return state

#路由
def intent_router(state:MyState)->Literal['weather', 'translate', 'chat', 'unknown']:
    return state["output"]

#构建图
builder_graph = StateGraph(MyState)
##添加node和Edge
builder_graph.add_node("llm_node",llm_node)
builder_graph.add_node("chat",chat_node)
builder_graph.add_node("weather",weather_node)
builder_graph.add_node("translate",translation_node)
builder_graph.add_node("unknown",unknown_node)

builder_graph.add_edge(START,"llm_node")
builder_graph.add_conditional_edges("llm_node",intent_router)
# 分支节点到 END 是“边”，不是“节点”
builder_graph.add_edge("chat",END)
builder_graph.add_edge("weather",END)
builder_graph.add_edge("translate",END)
builder_graph.add_edge("unknown",END)

#编译图和执行图
graph = builder_graph.compile()
graph.invoke({"input":"翻译一下这个:think"})



{'input': '翻译一下这个:think', 'output': '翻译...'}